# 第15章 动作识别

本章节学习目标：

- 理解本章核心算法的**数学原理**
- 掌握算法的**手写实现**方法
- 学会使用 OpenCV 对应函数进行**工程实践**
- 通过编程练习加深对算法的理解

> **📌 学习建议**：先阅读概念说明，再动手编写代码，最后完成练习


In [ ]:
# -*- coding: utf-8 -*-
# 中文路径兼容的图像读写函数
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False


# 代码实现

我们将开始动手实现一个C3D模型并在UCF101数据集上进行训练和测试。

In [ ]:
import torch.nn as nn

class C3D(nn.Module):
    """
    C3D，Convolution 3D，即三维卷积
    """

    def __init__(self):
        super(C3D, self).__init__()
        # 注意到核函数为三维，符合三维卷积的要求
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool4 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool5 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=(0, 1, 1))

        self.fc6 = nn.Linear(8192, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        self.fc8 = nn.Linear(4096, 487)

        self.dropout = nn.Dropout(p=0.5)

        self.relu = nn.ReLU()
        self.softmax = nn.Softmax()

    def forward(self, x):

        h = self.relu(self.conv1(x))
        h = self.pool1(h)

        h = self.relu(self.conv2(h))
        h = self.pool2(h)

        h = self.relu(self.conv3a(h))
        h = self.relu(self.conv3b(h))
        h = self.pool3(h)

        h = self.relu(self.conv4a(h))
        h = self.relu(self.conv4b(h))
        h = self.pool4(h)

        h = self.relu(self.conv5a(h))
        h = self.relu(self.conv5b(h))
        h = self.pool5(h)

        h = h.view(-1, 8192)
        h = self.relu(self.fc6(h))
        h = self.dropout(h)
        h = self.relu(self.fc7(h))
        h = self.dropout(h)

        logits = self.fc8(h)
        probs = self.softmax(logits)

        return probs

我们将基于C3D在UCF101数据集中实现视频动作识别。这里我们将直接调用已经完成的代码，你可以在GitHub链接中查看详细的项目。

In [ ]:
!git clone https://github.com/Niki173/C3D

在配置好UCF101数据集和模型预训练的权重之后，我们便可以进行C3D的训练。

In [ ]:
!python train.py

这里我们只展示部分训练流程。


<center>
    <span style="color:red">[图片占位 - 需本地生成]</span> width=1000>
    <br>
    <div style="color:orange; 
    display: inline-block;
    color: #999;
    padding: 2px;"></div>
</center>

最后，我们在测试集上进行测试，准确率可达 96\%。


<center>
    <span style="color:red">[图片占位 - 需本地生成]</span> width=1000>
    <br>
    <div style="color:orange; 
    display: inline-block;
    color: #999;
    padding: 2px;"></div>
</center>


<center>
    <span style="color:red">[图片占位 - 需本地生成]</span> width=600>
    <br>
    <div style="color:orange; 
    display: inline-block;
    color: #999;
    padding: 2px;"></div>
</center>

<!-- 目前的困难：
1. 姿态估计的精确度问题：由于人体姿态估计系统使用的是多种视觉信息，如果它们不能够准确地追踪到人体的每个部位，那么估计出来的结果也会不准确。
1. 尺度不变性问题：尽管现有的人体姿态估计算法可以处理不同尺度的图像，但它们仍然存在尺度不变性问题，即当姿态发生变化时，估计的结果可能会受到影响。
1. 光照变化问题：由于光照变化会对视觉信息产生影响，因此人体姿态估计系统可能无法准确地识别不同光照环境下的人体部位。
1. 复杂背景问题：复杂的背景可能会干扰人体姿态估计系统的性能，使得它无法准确地识别人体部位。 -->


---

## 📝 练习：本章算法手写实现与扩展


**练习目标**：基于本章所学内容，完成以下实践任务。

**要求**：
1. 手写实现本章的核心算法（不直接调用 OpenCV/PyTorch 对应函数）
2. 使用本章学习的方法处理至少 2 张不同的测试图像
3. 对比手写实现与现成库函数的结果差异
4. 分析算法参数对结果的影响
5. 撰写 200 字以上的实验报告


**💡 小提示**：
- 除 `cv_imread` / `cv_imwrite` 外，不直接调用 OpenCV 高层函数
- 使用 NumPy 进行矩阵运算
- 注意边界处理和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
```python
# 本章练习代码框架
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

# ============================================
# TODO: 在此处手写实现本章核心算法
# ============================================

# 示例框架：
# 1. 数据准备
# img = cv_imread('test_image.jpg')
# gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. 手写算法实现
# def algorithm_manual(input_image, **params):
#     # TODO: 实现算法核心逻辑
#     # 要求：除 OpenCV 读写函数外，其余代码手写
#     return output

# 3. 对比验证
# result_manual = algorithm_manual(gray)
# result_library = cv2.XXX(gray)  # 对应库函数
# diff = np.abs(result_manual.astype(float) - result_library.astype(float))
# print(f"最大差异: {diff.max()}")

# 4. 参数敏感性分析
# for param in [param1, param2, param3]:
#     result = algorithm_manual(gray, param=param)
#     # 可视化结果变化

# 5. 实验报告
print("请完成上述练习并撰写实验报告")
```



### 💻 代码要点解释

1. **图像读取与保存**：使用自定义的 `cv_imread` / `cv_imwrite` 函数，解决 Windows 中文路径下 OpenCV 读写图像失败的问题

2. **算法核心**：手写实现的核心在于**不依赖现成库函数**，而是直接操作像素和矩阵运算

3. **对比验证**：通过与 OpenCV 对应函数的结果进行数值对比，验证手写实现的正确性

4. **参数分析**：调整算法参数，观察输出变化，理解每个参数的物理含义

5. **扩展思考**：尝试将算法应用到自己的图像上，或改进算法（如增加加速技巧）

---

</details>

---
